# UIT DSC 2026 – LegalQA Pipeline Smoke Test 30 ID

Notebook kiểm tra **pipeline cuối cùng** gồm BM25 + Dense + RRF + Reranker và ba route `generated_512`, `extractive_long`, `extractive_fallback`.

Notebook này **không có lệnh build BM25/Dense**. Thiếu cache sẽ dừng ngay, không chạy tiền xử lý nhiều giờ.

Trước khi chạy trên Kaggle:

1. Bật GPU và Internet.
2. Add Input dataset LegalQA chứa `public-official.json`.
3. Add Input phần Notebook Output của `uit-dsc-2026-subtask2`, version `346318039`.
4. Output cũ phải có `legalqa.sqlite`, `legalqa_dense.meta.json`, `legalqa_dense.faiss` hoặc `.npy`, và ba model snapshot.

Tập test gồm ID lỗi đã biết, câu dài theo router và câu bình thường lấy ngẫu nhiên. Output dùng tên riêng và không resume submission cũ.

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import shutil
import sqlite3
import statistics
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from IPython.display import FileLink, Markdown, display

REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"
SMOKE_LIMIT, RISKY_TARGET, SEED = 30, 15, 2026
PRIORITY_IDS = ["34235", "62147", "86293", "80189", "135669"]
HISTORICAL_TOKEN_LIMIT_IDS = ["97935", "83665", "63093", "76135", "94131", "120127", "103999", "59823"]
RETRIEVAL_MEDIAN_MAX_SECONDS = 2.0
SMOKE_MEDIAN_MAX_SECONDS = 15.0
# Sau vòng retrieval đầu tiên, điền document_id đúng cho đủ 5 known problem rồi chạy lại.
RETRIEVAL_EXPECTED_DOCUMENT_IDS = {
    # "86293": "document_id đọc từ bảng Top-3/20/50",
}
# Sau khi đọc nguyên văn 5 known_problem + 10 random ở cuối notebook, điền đủ 15 ID.
MANUAL_REVIEW_APPROVED_IDS = []

MODE, KNN_THRESHOLD = "rag", 0.72
BM25_TOP_K, DENSE_TOP_K, RRF_K, RRF_TOP_K = 50, 50, 60, 50
RERANKER_CANDIDATE_K, RERANK_TOP_K = 20, 3
DENSE_QUERY_MAX_LENGTH, RERANKER_MAX_LENGTH = 256, 1024
MAX_NEW_TOKENS, MAX_INPUT_TOKENS = 512, 7168
REPETITION_PENALTY, MIN_LLM_ANSWER_TOKENS = 1.05, 8
GENERATION_SEED, DEVICE = 2026, "auto"

EMBEDDING_MODEL_ID = "AITeamVN/Vietnamese_Embedding_v2"
RERANKER_MODEL_ID = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"
MODEL_MARKER = ".legalqa_model.json"

if Path("/kaggle/working").is_dir():
    PLATFORM = "Kaggle"
    INPUT_ROOT = Path("/kaggle/input")
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    WORK_DIR = Path("/kaggle/working/legalqa-smoke30")
    EXPORT_DIR = Path("/kaggle/working")
elif Path("/content").is_dir():
    PLATFORM = "Colab"
    INPUT_ROOT = Path("/content")
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    WORK_DIR = Path("/content/legalqa-smoke30")
    EXPORT_DIR = WORK_DIR
else:
    PLATFORM = "Local"
    INPUT_ROOT = Path(".").resolve()
    REPO_DIR = Path(".").resolve()
    WORK_DIR = REPO_DIR / "artifacts-smoke30"
    EXPORT_DIR = WORK_DIR

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK_DIR / "hf-cache")
os.environ["HF_HUB_CACHE"] = str(WORK_DIR / "hf-cache/hub")
print(f"Platform: {PLATFORM} | Input: {INPUT_ROOT} | Work: {WORK_DIR}")
print("Mode: inference-only; không build BM25/Dense.")

## 1. Clone code mới nhất và cài dependency

In [ ]:
def run(command: list[str], cwd: Path | None = None) -> None:
    print("$", " ".join(map(str, command)))
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, encoding="utf-8", errors="replace",
    )
    if process.stdout is not None:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.stdout.close()
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)

if PLATFORM != "Local":
    if (REPO_DIR / ".git").is_dir():
        run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)
    else:
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print(f"Code commit: {commit}")
requirements = REPO_DIR / "requirements-generator.txt"
if requirements.is_file():
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    run([sys.executable, "-m", "pip", "install", "-q", "torch",
         "transformers>=4.40.0", "accelerate", "sentencepiece",
         "faiss-cpu", "numpy", "tqdm"])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch
from legalqa_baseline.text import (
    clean_answer, is_long_form_question, is_refusal_answer, possibly_cut,
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA không khả dụng. Hãy bật GPU Accelerator trên Kaggle.")
print(f"CUDA: {torch.cuda.device_count()} GPU(s)")
for gpu_id in range(torch.cuda.device_count()):
    print(f"  GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")

# Nhóm test này rất nhanh và xác nhận prompt/router/fallback của commit vừa clone.
# Dùng discovery vì thư mục tests của repo không có __init__.py, nên không thể import
# theo dạng tests.test_generator trên Kaggle/Python 3.12.
for test_file in ("test_generator.py", "test_long_answer_routing.py"):
    run([
        sys.executable, "-m", "unittest", "discover",
        "-s", "tests", "-p", test_file, "-v",
    ], cwd=REPO_DIR)

## 2. Tìm và xác minh cache

Thiếu index/model sẽ dừng ngay. Cell này không có nhánh build lại.

In [ ]:
def artifact_rank(path: Path) -> tuple[int, int, str]:
    has_manifest = any((parent / "legalqa_artifacts.json").is_file() for parent in path.parents)
    priority = 0 if has_manifest else (1 if "legalqa-run" in path.parts else 2)
    return priority, len(path.parts), str(path)

INPUT_FILES = [p for p in INPUT_ROOT.rglob("*") if p.is_file()]
print(f"Đã quét {len(INPUT_FILES):,} file trong Kaggle Input.")

def candidates(names: set[str]) -> list[Path]:
    accepted = {name.casefold() for name in names}
    return sorted(
        (p for p in INPUT_FILES if p.name.casefold() in accepted),
        key=artifact_rank,
    )

def require_file(label: str, names: set[str]) -> Path:
    found = candidates(names)
    if not found:
        raise FileNotFoundError(
            f"Thiếu {label}: {sorted(names)}. Add Input dataset và output version 346318039."
        )
    return found[0]

def complete_model(folder: Path) -> bool:
    return (folder / "config.json").is_file() and (
        any(folder.glob("*.safetensors")) or any(folder.glob("pytorch_model*.bin"))
    )

def discover_models() -> dict[str, Path]:
    found: dict[str, list[Path]] = {}
    for marker in (p for p in INPUT_FILES if p.name == MODEL_MARKER):
        try:
            payload = json.loads(marker.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        repo_id = payload.get("repo_id") if isinstance(payload, dict) else None
        if isinstance(repo_id, str) and complete_model(marker.parent):
            found.setdefault(repo_id, []).append(marker.parent)
    return {repo: sorted(paths, key=artifact_rank)[0] for repo, paths in found.items()}

PUBLIC_PATH = require_file(
    "public test",
    {"public-official.json", "public-official(1).json", "public_official.json", "public_test.json"},
)
DB_PATH = require_file("BM25 SQLite index", {"legalqa.sqlite"})
DENSE_META_PATH = require_file("Dense metadata", {"legalqa_dense.meta.json"})
DENSE_INDEX_PATH = DENSE_META_PATH.with_name("legalqa_dense")
if not (DENSE_INDEX_PATH.with_suffix(".faiss").is_file() or DENSE_INDEX_PATH.with_suffix(".npy").is_file()):
    raise FileNotFoundError(f"Thiếu legalqa_dense.faiss/.npy cạnh {DENSE_META_PATH}")

with sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True) as connection:
    db_meta = dict(connection.execute("SELECT key, value FROM metadata"))
if db_meta.get("schema_version") != "5" or db_meta.get("corpus_hash_version") != "2" or int(db_meta.get("chunks", 0)) <= 0:
    raise RuntimeError(f"BM25 index không tương thích: {db_meta}")

# Không query SQLite FTS trực tiếp trên /kaggle/input: random I/O có thể
# làm mỗi câu chậm hàng trăm giây. Sao chép một lần sang local working disk.
if PLATFORM == "Kaggle" and INPUT_ROOT in DB_PATH.resolve().parents:
    source_db = DB_PATH
    local_db = WORK_DIR / "legalqa.sqlite"
    if not local_db.is_file() or local_db.stat().st_size != source_db.stat().st_size:
        copying_db = WORK_DIR / "legalqa.sqlite.copying"
        copying_db.unlink(missing_ok=True)
        print(f"Copy BM25 SQLite vào local disk ({source_db.stat().st_size / 1024**3:.2f} GiB)...")
        shutil.copyfile(source_db, copying_db)
        if copying_db.stat().st_size != source_db.stat().st_size:
            raise IOError("Bản sao SQLite bị thiếu dữ liệu")
        copying_db.replace(local_db)
    DB_PATH = local_db
    print(f"BM25 local: {DB_PATH}")

dense_payload = json.loads(DENSE_META_PATH.read_text(encoding="utf-8"))
dense_meta = dense_payload.get("manifest", {}) if isinstance(dense_payload, dict) else {}
if (int(dense_meta.get("schema_version", 0)) < 5
        or dense_meta.get("corpus_hash_version") != "2"
        or dense_meta.get("pooling") != "cls"
        or dense_meta.get("normalization") != "l2"
        or dense_meta.get("similarity") != "dot_product"):
    raise RuntimeError(f"Dense index không tương thích: {dense_meta}")

models = discover_models()
required = [EMBEDDING_MODEL_ID, RERANKER_MODEL_ID, GENERATOR_MODEL_ID]
missing = [repo for repo in required if repo not in models]
if missing:
    raise FileNotFoundError("Thiếu model snapshot trong Kaggle Input: " + ", ".join(missing))
EMBEDDING_MODEL = str(models[EMBEDDING_MODEL_ID])
RERANKER_MODEL = str(models[RERANKER_MODEL_ID])
GENERATOR_MODEL = str(models[GENERATOR_MODEL_ID])

public_data = json.loads(PUBLIC_PATH.read_text(encoding="utf-8"))
if not isinstance(public_data, dict) or not public_data:
    raise ValueError("Public test phải là JSON object không rỗng")
if not all(isinstance(x, dict) and isinstance(x.get("question"), str) for x in public_data.values()):
    raise ValueError("Public test có item sai schema")

print(f"Public: {PUBLIC_PATH} ({len(public_data):,} ID)")
print(f"BM25: {DB_PATH} | chunks={int(db_meta.get('chunks', 0)):,}")
print(f"Dense: {DENSE_INDEX_PATH}")
for repo in required:
    print(f"Model {repo}: {models[repo]}")
print("[✓] Đủ cache — không chạy preprocessing.")

## 3. Chọn 30 ID đại diện

In [ ]:
selected_ids: list[str] = []
reason: dict[str, str] = {}

def add(qid: str, why: str) -> None:
    qid = str(qid)
    if qid in public_data and qid not in selected_ids and len(selected_ids) < SMOKE_LIMIT:
        selected_ids.append(qid)
        reason[qid] = why

for qid in PRIORITY_IDS:
    add(qid, "known_problem")
for qid in HISTORICAL_TOKEN_LIMIT_IDS:
    add(qid, "historical_token_limit")

long_ids = [
    str(qid) for qid, item in public_data.items()
    if is_long_form_question(str(item.get("question") or ""))
]
long_ids.sort(key=lambda qid: len(str(public_data[qid].get("question") or "")), reverse=True)
for qid in long_ids:
    if len(selected_ids) >= RISKY_TARGET:
        break
    add(qid, "long_form")

normal_pool = [
    str(qid) for qid, item in public_data.items()
    if str(qid) not in selected_ids
    and not is_long_form_question(str(item.get("question") or ""))
]
rng = random.Random(SEED)
rng.shuffle(normal_pool)
for qid in normal_pool:
    add(qid, "random_normal")
    if len(selected_ids) >= SMOKE_LIMIT:
        break
if len(selected_ids) != SMOKE_LIMIT:
    raise RuntimeError(f"Chỉ chọn được {len(selected_ids)}/{SMOKE_LIMIT} ID")

smoke_data = {qid: public_data[qid] for qid in selected_ids}
SMOKE_INPUT_PATH = WORK_DIR / "public_smoke30.json"
SMOKE_INPUT_PATH.write_text(json.dumps(smoke_data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Selection:", dict(Counter(reason.values())))
rows = ["| # | ID | Nhóm | Long router | Câu hỏi |", "|---:|---|---|:---:|---|"]
for index, qid in enumerate(selected_ids, 1):
    question = str(public_data[qid]["question"]).replace("|", "&#124;")
    question = question if len(question) <= 120 else question[:117] + "…"
    long_flag = "✓" if is_long_form_question(public_data[qid]["question"]) else ""
    rows.append(f"| {index} | {qid} | {reason[qid]} | {long_flag} | {question} |")
display(Markdown("\n".join(rows)))
print(f"Input 30 ID: {SMOKE_INPUT_PATH}")

## 4. Vòng 1 retrieval-only, sau đó mới chạy smoke30 đầy đủ

Vòng retrieval-only chạy đúng BM25 + Dense + RRF + Reranker nhưng không gọi generator. Notebook chỉ sang vòng sinh đáp án khi median retrieval không quá 2 giây/câu và document đúng của đủ 5 known problem còn hiện diện ở Top-50, Top-20 và Top-3. Điền `RETRIEVAL_EXPECTED_DOCUMENT_IDS` từ bảng chẩn đoán sau lần chạy đầu.

In [ ]:
RETRIEVAL_PATH = WORK_DIR / "retrieval_smoke30.json"
RETRIEVAL_GATE_PATH = WORK_DIR / "retrieval_gate.json"
SUBMISSION_PATH = WORK_DIR / "submission_smoke30.json"
AUDIT_PATH = WORK_DIR / "submission_smoke30.audit.jsonl"
CHECKPOINT_PATH = SUBMISSION_PATH.with_suffix(".checkpoint.json")
for stale in (RETRIEVAL_PATH, RETRIEVAL_GATE_PATH, SUBMISSION_PATH, AUDIT_PATH, CHECKPOINT_PATH):
    stale.unlink(missing_ok=True)

common_retrieval_args = [
    "--input", str(SMOKE_INPUT_PATH), "--db", str(DB_PATH),
    "--knn-threshold", str(KNN_THRESHOLD),
    "--context-top-k", str(RERANK_TOP_K),
    "--bm25-top-k", str(BM25_TOP_K), "--dense-top-k", str(DENSE_TOP_K),
    "--rrf-k", str(RRF_K), "--rrf-top-k", str(RRF_TOP_K),
    "--reranker-candidate-k", str(RERANKER_CANDIDATE_K),
    "--rerank-top-k", str(RERANK_TOP_K),
    "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
    "--reranker-max-length", str(RERANKER_MAX_LENGTH),
    "--dense-index", str(DENSE_INDEX_PATH),
    "--embedding-model", EMBEDDING_MODEL, "--reranker-model", RERANKER_MODEL,
    "--device", DEVICE,
]
retrieval_cmd = [
    sys.executable, "-m", "legalqa_baseline", "diagnose-retrieval",
    *common_retrieval_args, "--output", str(RETRIEVAL_PATH),
]
run(retrieval_cmd, cwd=REPO_DIR)
retrieval_payload = json.loads(RETRIEVAL_PATH.read_text(encoding="utf-8"))
retrieval_items = {str(item["id"]): item for item in retrieval_payload["items"]}
retrieval_median = float(retrieval_payload["summary"]["median_stage_seconds"]["total"])

def document_rank(qid: str, stage: str, expected_document_id: str) -> int | None:
    candidates = retrieval_items[qid].get("diagnostic_candidates", {}).get(stage, [])
    for candidate in candidates:
        candidate_ids = {str(candidate.get("document_id") or ""), str(candidate.get("context_id") or "")}
        if str(expected_document_id) in candidate_ids:
            return int(candidate["rank"])
    return None

retrieval_checks = {}
missing_expected_ids = []
for qid in PRIORITY_IDS:
    expected = RETRIEVAL_EXPECTED_DOCUMENT_IDS.get(qid)
    if not expected:
        missing_expected_ids.append(qid)
        continue
    retrieval_checks[qid] = {
        stage: document_rank(qid, stage, str(expected))
        for stage in ("top50", "top20", "top3")
    }
retrieval_speed_gate_pass = retrieval_median <= RETRIEVAL_MEDIAN_MAX_SECONDS
retrieval_document_gate_pass = (
    not missing_expected_ids
    and all(all(rank is not None for rank in ranks.values()) for ranks in retrieval_checks.values())
)
retrieval_gate = {
    "median_seconds": retrieval_median,
    "median_limit_seconds": RETRIEVAL_MEDIAN_MAX_SECONDS,
    "speed_gate_pass": retrieval_speed_gate_pass,
    "document_gate_pass": retrieval_document_gate_pass,
    "missing_expected_ids": missing_expected_ids,
    "known_problem_ranks": retrieval_checks,
}
retrieval_gate["pass"] = retrieval_speed_gate_pass and retrieval_document_gate_pass
RETRIEVAL_GATE_PATH.write_text(json.dumps(retrieval_gate, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(retrieval_gate, ensure_ascii=False, indent=2))
retrieval_rows = ["| ID | Stage | Rank | Document | Title | Preview |", "|---|---|---:|---|---|---|"]
for qid in PRIORITY_IDS:
    for stage in ("top3", "top20", "top50"):
        for candidate in retrieval_items[qid].get("diagnostic_candidates", {}).get(stage, [])[:3]:
            title = str(candidate.get("title") or "").replace("|", "&#124;")
            preview = str(candidate.get("text_preview") or "").replace("|", "&#124;")[:180]
            retrieval_rows.append(f"| {qid} | {stage} | {candidate['rank']} | {candidate.get('document_id')} | {title} | {preview} |")
display(Markdown("\n".join(retrieval_rows)))
if not retrieval_gate["pass"]:
    raise RuntimeError(
        "Retrieval gate chưa đạt. Mở retrieval_smoke30.json, điền đúng document_id "
        "cho đủ 5 PRIORITY_IDS vào RETRIEVAL_EXPECTED_DOCUMENT_IDS rồi chạy lại; "
        "không chạy generator/full smoke khi retrieval chưa đạt."
    )

cmd = [
    sys.executable, "-m", "legalqa_baseline", "predict",
    *common_retrieval_args, "--output", str(SUBMISSION_PATH),
    "--audit-output", str(AUDIT_PATH), "--mode", MODE,
    "--generator-model", GENERATOR_MODEL,
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--max-input-tokens", str(MAX_INPUT_TOKENS),
    "--repetition-penalty", str(REPETITION_PENALTY),
    "--min-llm-answer-tokens", str(MIN_LLM_ANSWER_TOKENS),
    "--generation-seed", str(GENERATION_SEED), "--checkpoint-interval", "1",
]
print(f"Retrieval gate PASS; chạy smoke30 với reranker Top-{RERANKER_CANDIDATE_K}/{RRF_TOP_K}.")
run(cmd, cwd=REPO_DIR)
if not SUBMISSION_PATH.is_file() or not AUDIT_PATH.is_file():
    raise RuntimeError("Pipeline kết thúc nhưng thiếu submission/audit")
print(f"[✓] Submission: {SUBMISSION_PATH}")
print(f"[✓] Audit: {AUDIT_PATH}")

## 5. Audit output cuối

`generation_*` mô tả lỗi model trước fallback. `final_*` được tính lại từ câu trả lời cuối trong submission.

In [ ]:
predictions = json.loads(SUBMISSION_PATH.read_text(encoding="utf-8"))
audit = [json.loads(line) for line in AUDIT_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
audit_by_id = {str(item["id"]): item for item in audit}
if set(predictions) != set(smoke_data) or set(audit_by_id) != set(smoke_data):
    raise AssertionError("ID mismatch giữa input, prediction hoặc audit")
if not all(isinstance(x, dict) and set(x) == {"answer"} and str(x["answer"]).strip() for x in predictions.values()):
    raise AssertionError("Submission có answer rỗng hoặc sai schema")

answers = {qid: predictions[qid]["answer"].strip() for qid in selected_ids}
routes = Counter(str(x.get("route")) for x in audit)
boilerplate_ids = [qid for qid, ans in answers.items() if clean_answer(ans) != ans]
no_info_ids = [qid for qid, ans in answers.items() if is_refusal_answer(ans)]
cut_ids = [qid for qid, ans in answers.items() if possibly_cut(ans)]
generation_hit_ids = [qid for qid, x in audit_by_id.items() if x.get("hit_token_limit")]
final_token_limited_ids = [qid for qid in generation_hit_ids if audit_by_id[qid].get("route") != "extractive_fallback"]
successful_token_fallback_ids = [qid for qid in generation_hit_ids if audit_by_id[qid].get("route") == "extractive_fallback"]
lengths = sorted(len(x.split()) for x in answers.values())
stage_medians = {}
for stage in ("bm25", "dense", "fusion", "reranker", "generation", "total"):
    values = [float(x.get("stage_seconds", {}).get(stage, 0.0)) for x in audit]
    stage_medians[stage] = round(statistics.median(values), 3) if values else 0.0
full_speed_gate_pass = stage_medians["total"] <= SMOKE_MEDIAN_MAX_SECONDS
missing_historical_token_ids = [qid for qid in HISTORICAL_TOKEN_LIMIT_IDS if qid not in audit_by_id]
historical_token_case_outcomes = {
    qid: {"route": audit_by_id[qid].get("route"), "hit_token_limit": bool(audit_by_id[qid].get("hit_token_limit"))}
    for qid in HISTORICAL_TOKEN_LIMIT_IDS if qid in audit_by_id
}
historical_token_case_failures = [
    *missing_historical_token_ids,
    *[qid for qid in HISTORICAL_TOKEN_LIMIT_IDS if qid in audit_by_id
    and (qid in final_token_limited_ids or qid in boilerplate_ids or qid in no_info_ids or qid in cut_ids)
    ],
]

summary = {
    "ids": len(predictions),
    "routes": dict(routes),
    "words_min": min(lengths),
    "words_median": statistics.median(lengths),
    "words_p90": lengths[round(0.9 * (len(lengths) - 1))],
    "words_max": max(lengths),
    "reranker_candidate_k": RERANKER_CANDIDATE_K,
    "reranker_max_length": RERANKER_MAX_LENGTH,
    "stage_median_seconds": stage_medians,
    "median_total_limit_seconds": SMOKE_MEDIAN_MAX_SECONDS,
    "speed_gate_pass": full_speed_gate_pass,
    "generation_hit_token_limit_count": len(generation_hit_ids),
    "successful_token_fallback_ids": successful_token_fallback_ids,
    "historical_token_limit_case_count": len(HISTORICAL_TOKEN_LIMIT_IDS) - len(missing_historical_token_ids),
    "historical_token_case_outcomes": historical_token_case_outcomes,
    "missing_historical_token_ids": missing_historical_token_ids,
    "historical_token_case_failures": historical_token_case_failures,
    "generation_refusal_trigger_count": sum(bool(x.get("says_no_information")) for x in audit),
    "final_token_limited_ids": final_token_limited_ids,
    "final_boilerplate_ids": boilerplate_ids,
    "final_no_information_ids": no_info_ids,
    "final_possibly_cut_ids": cut_ids,
}
summary["mechanical_gate_pass"] = not any((final_token_limited_ids, boilerplate_ids, no_info_ids, cut_ids))
summary["automatic_smoke_gate_pass"] = (
    summary["mechanical_gate_pass"]
    and full_speed_gate_pass
    and not historical_token_case_failures
)

items = []
rows = [
    "| ID | Nhóm | Route | Words | BM25 s | Dense s | Rerank s | Gen s | Total s | Gen tokens | Hit | Final flags |",
    "|---|---|---|---:|---:|---:|---:|---:|---:|---:|:---:|---|",
]
for qid in selected_ids:
    record, flags = audit_by_id[qid], []
    if qid in boilerplate_ids: flags.append("boilerplate")
    if qid in no_info_ids: flags.append("no_info")
    if qid in cut_ids: flags.append("possibly_cut")
    if qid in final_token_limited_ids: flags.append("token_limited")
    item = {
        "id": qid, "selection_reason": reason[qid],
        "question": public_data[qid]["question"], "route": record.get("route"),
        "answer_words": len(answers[qid].split()), "generated_tokens": record.get("generated_tokens"),
        "hit_token_limit": bool(record.get("hit_token_limit")),
        "generation_says_no_information": bool(record.get("says_no_information")),
        "final_flags": flags, "top_document_id": record.get("top_document_id"),
        "reranker_score": record.get("reranker_score"),
        "stage_seconds": record.get("stage_seconds", {}),
    }
    items.append(item)
    rows.append(
        f"| {qid} | {reason[qid]} | {record.get('route')} | {item['answer_words']} | "
        f"{item['stage_seconds'].get('bm25', 0):.2f} | {item['stage_seconds'].get('dense', 0):.2f} | "
        f"{item['stage_seconds'].get('reranker', 0):.2f} | {item['stage_seconds'].get('generation', 0):.2f} | "
        f"{item['stage_seconds'].get('total', 0):.2f} | {record.get('generated_tokens') or 0} | "
        f"{'✓' if record.get('hit_token_limit') else ''} | {', '.join(flags) or 'OK'} |"
    )

REVIEW_PATH = WORK_DIR / "pipeline_smoke30_review.json"
REVIEW_PATH.write_text(json.dumps({"summary": summary, "items": items}, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(Markdown("\n".join(rows)))
print("[✓] Automatic smoke gate PASS — chuyển sang kiểm tra tay 15 câu." if summary["automatic_smoke_gate_pass"] else "[!] Automatic smoke gate FAIL — chưa được chạy full 1.000 câu.")

## 6. Preview theo route và xuất kết quả

In [ ]:
random_review_ids = [qid for qid in selected_ids if reason[qid] == "random_normal"][:10]
preview_ids = list(dict.fromkeys([*PRIORITY_IDS, *random_review_ids]))
required_manual_review_ids = set(preview_ids)
approved_manual_review_ids = {str(qid) for qid in MANUAL_REVIEW_APPROVED_IDS}
missing_manual_review_ids = sorted(required_manual_review_ids - approved_manual_review_ids)

for qid in preview_ids:
    question, answer = public_data[qid]["question"], predictions[qid]["answer"]
    route = audit_by_id[qid].get("route")
    display(Markdown(f"### ID {qid} — `{route}`\n**Câu hỏi:** {question}\n\n**Trả lời nguyên văn ({len(answer.split())} từ):**\n\n{answer}"))

FULL_GATE_PATH = WORK_DIR / "full_run_gate.json"
full_gate = {
    "retrieval_gate_pass": bool(retrieval_gate["pass"]),
    "mechanical_gate_pass": bool(summary["mechanical_gate_pass"]),
    "speed_gate_pass": bool(summary["speed_gate_pass"]),
    "automatic_smoke_gate_pass": bool(summary["automatic_smoke_gate_pass"]),
    "required_manual_review_ids": sorted(required_manual_review_ids),
    "approved_manual_review_ids": sorted(approved_manual_review_ids & required_manual_review_ids),
    "missing_manual_review_ids": missing_manual_review_ids,
}
full_gate["full_1000_unlocked"] = (
    full_gate["retrieval_gate_pass"]
    and full_gate["automatic_smoke_gate_pass"]
    and not missing_manual_review_ids
)
FULL_GATE_PATH.write_text(json.dumps(full_gate, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(full_gate, ensure_ascii=False, indent=2))
if full_gate["full_1000_unlocked"]:
    print("[✓] FULL 1.000 UNLOCKED — tất cả gate đã đạt.")
else:
    print("[!] FULL 1.000 LOCKED — không chạy full cho tới khi mọi gate và 15 review tay đều đạt.")

paths = [SMOKE_INPUT_PATH, RETRIEVAL_PATH, RETRIEVAL_GATE_PATH, SUBMISSION_PATH, AUDIT_PATH, REVIEW_PATH, FULL_GATE_PATH]
if CHECKPOINT_PATH.is_file(): paths.append(CHECKPOINT_PATH)
final_paths = []
for source in paths:
    destination = EXPORT_DIR / source.name
    if destination.resolve() != source.resolve(): shutil.copy2(source, destination)
    final_paths.append(destination)

BUNDLE_PATH = EXPORT_DIR / "legalqa_pipeline_smoke30_results.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in final_paths: archive.write(path, arcname=path.name)
for path in [*final_paths, BUNDLE_PATH]:
    print(" -", path)
display(FileLink(str(EXPORT_DIR / SUBMISSION_PATH.name)))
display(FileLink(str(EXPORT_DIR / AUDIT_PATH.name)))
display(FileLink(str(EXPORT_DIR / REVIEW_PATH.name)))
display(FileLink(str(BUNDLE_PATH)))

## Cách kết luận

- Vòng 1 chỉ PASS khi median retrieval ≤ 2 giây/câu và document đúng của cả 5 known problem xuất hiện ở Top-50, Top-20, Top-3.
- Vòng 2 yêu cầu `mechanical_gate_pass=true`, không final refusal/cắt, mọi lần chạm token đều fallback và median tổng ≤ 15 giây/câu.
- Notebook hiển thị nguyên văn 5 known problem + 10 random. Chỉ thêm ID vào `MANUAL_REVIEW_APPROVED_IDS` sau khi đã đọc và xác nhận đúng tài liệu/nội dung.
- Chỉ khi `full_1000_unlocked=true` mới chạy full 1.000 câu. Nếu cache không được nhận diện, kiểm tra Add Input của version `346318039`; không build lại trong notebook smoke.